# Stage 2: MLP classifier — Whisper embedding vs Mel-spectrogram
Same MLP, same training recipe, same split → the only difference is the input feature.

| method | input to MLP | dim |
|---|---|---|
| **1. Whisper** | `whisper_pool.npy` (mean of Whisper encoder frames over valid audio) | 768 |
| **2. Mel** | statistics pooling of `mel.npy` over time: mean + std of log-mel, std of Δ (padding frames excluded) | 384 |
| *(extra)* **whisper+mel** | concatenation of 1 and 2 | 1152 |

**Label set:** the source dataset's *train* split has only **9 accents (~1000 clips each, balanced)**, while dev/test keep the original Common Voice distribution and contain 7 extra accents never seen in training. We therefore use the 9 train accents as the class set and drop val/test clips of unseen accents (they can't be predicted by any classifier trained here).

**Prior shift:** train is balanced, but val/test are not (`us` ≈ 47%). After training we apply **logit adjustment** `log p(y|x) + τ·log π_val(y)`, with τ tuned on val (val has the same distribution as test).

Why pool the Mel instead of flattening it? Flattening `[128 × 376]` = 48k inputs → huge first layer, overfits, and is not shift-invariant in time. Pooling gives a fixed-size, time-invariant summary that an MLP can actually use.

Run on Kaggle: add the **output of notebook 01** (folder with `splits.csv`, `mel.npy`, `whisper_pool.npy`) as Input. Locally: set `DATA_DIR`.

In [ ]:
# !pip install -q scikit-learn pandas matplotlib

## 0. Config

In [ ]:
import json, time, copy, itertools
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt

# --- where notebook 01 wrote its outputs (searched for splits.csv) ---
DATA_DIR = None   # e.g. Path('data') locally; None = auto-search
if DATA_DIR is None:
    for root in [Path('/kaggle/input'), Path('/kaggle/working'), Path('data'), Path('../data')]:
        hits = sorted(root.rglob('splits.csv')) if root.exists() else []
        if hits: DATA_DIR = hits[0].parent; break
assert DATA_DIR is not None and (Path(DATA_DIR)/'splits.csv').exists(), 'splits.csv not found - set DATA_DIR'
DATA_DIR = Path(DATA_DIR)
OUT_DIR = Path('/kaggle/working/mlp') if Path('/kaggle/working').exists() else Path('mlp_out')
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('DATA_DIR =', DATA_DIR, '| OUT_DIR =', OUT_DIR)

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device:', DEV)

# --- MLP + training (shared by both methods) ---
CFG = dict(
    hidden=[512, 256],     # hidden layer sizes
    dropout=0.3,
    lr=1e-3, weight_decay=1e-2,
    batch_size=256, epochs=150, patience=20,   # early stopping on val macro-F1
    label_smoothing=0.1,
    class_weight=False,    # train split is already balanced (~1000/class)
    mixup_alpha=0.2,       # 0 = off
    input_noise=0.05,      # gaussian noise on standardized inputs, 0 = off
)
SEEDS = [0, 1, 2, 3, 4]    # final runs: mean ± std over seeds + seed-ensemble
RUN_SWEEP = True           # random search on val (section 4)
N_TRIALS = 20              # configs per feature set

## 1. Labels and splits

In [ ]:
df = pd.read_csv(DATA_DIR/'splits.csv')
display(df.groupby(['label', 'split']).size().unstack().fillna(0).astype(int)[['train', 'val', 'test']])

# class set = accents present in TRAIN; val/test clips of unseen accents are dropped
CLASSES = sorted(df.loc[df.split == 'train', 'label'].unique())
C = len(CLASSES); lab2id = {c: i for i, c in enumerate(CLASSES)}
y_all = df.label.map(lab2id).fillna(-1).astype(np.int64).values
unseen = df[y_all < 0]
print(f'dropped {len(unseen)} val/test clips of accents not in train:', dict(unseen.groupby("split").label.value_counts()))
IDX = {s: np.where((df.split.values == s) & (y_all >= 0))[0] for s in ('train', 'val', 'test')}
print(C, 'classes:', CLASSES); print({s: len(i) for s, i in IDX.items()})
PRIOR_VAL = np.bincount(y_all[IDX['val']], minlength=C) / len(IDX['val'])
print('val prior:', dict(zip(CLASSES, PRIOR_VAL.round(3))))

## 2. Features
### Method 1 — Whisper embedding
`whisper_pool.npy` is already one vector per clip (mean over the non-padded encoder frames).

In [ ]:
X_whisper = np.load(DATA_DIR/'whisper_pool.npy').astype(np.float32)
assert len(X_whisper) == len(df)
print('whisper', X_whisper.shape)

### Method 2 — Mel-spectrogram (statistics pooling)
Per mel band, over **valid** time frames only: mean, std of log-mel, and std of the time-difference (Δ, captures dynamics / speaking rhythm) → `128 × 3 = 384`.
Zero-padded frames become a flat column after `power_to_db(top_db=80)`, so frames whose spectrum is constant across bands are dropped before pooling (same idea as the valid-frame mean used for Whisper).

In [ ]:
def mel_stats_pool(mel_path, chunk=512):
    M = np.load(mel_path, mmap_mode='r')                # [N, n_mels, T] float16
    out = []
    for s in range(0, len(M), chunk):
        x = np.asarray(M[s:s+chunk], dtype=np.float32)  # [B, F, T]
        valid = np.ptp(x, axis=1) > 1e-3                # [B, T]  padded frames are flat
        valid[valid.sum(1) < 2] = True                  # all-silent clip -> use every frame
        w = valid[:, None, :].astype(np.float32); n = w.sum(-1)
        mean = (x * w).sum(-1) / n
        std  = np.sqrt((((x - mean[..., None]) ** 2) * w).sum(-1) / n)
        d    = np.diff(x, axis=-1); wd = w[..., 1:] * w[..., :-1]; nd = np.maximum(wd.sum(-1), 1)
        dmu  = (d * wd).sum(-1) / nd
        dstd = np.sqrt((((d - dmu[..., None]) ** 2) * wd).sum(-1) / nd)
        out.append(np.concatenate([mean, std, dstd], 1))
    return np.concatenate(out).astype(np.float32)

t0 = time.time()
X_mel = mel_stats_pool(DATA_DIR/'mel.npy')
assert len(X_mel) == len(df)
print('mel', X_mel.shape, f'({time.time()-t0:.1f}s)')

FEATURES = {'whisper': X_whisper, 'mel': X_mel, 'whisper+mel': np.concatenate([X_whisper, X_mel], 1)}

## 3. MLP + training loop
`[Linear → BatchNorm → GELU → Dropout] × len(hidden) → Linear(C)`.
Inputs are standardized with **train-only** mean/std (stored inside the model as buffers, so a saved model works on raw features).
Model selection = best **val macro-F1** (accuracy alone is misleading with imbalanced accents).

In [ ]:
class MLP(nn.Module):
    def __init__(self, d_in, n_cls, hidden=(512, 256), dropout=0.3, mu=None, sd=None):
        super().__init__()
        self.register_buffer('mu', torch.zeros(d_in) if mu is None else torch.as_tensor(mu))
        self.register_buffer('sd', torch.ones(d_in) if sd is None else torch.as_tensor(sd))
        layers, d = [], d_in
        for h in hidden:
            layers += [nn.Linear(d, h), nn.BatchNorm1d(h), nn.GELU(), nn.Dropout(dropout)]; d = h
        layers.append(nn.Linear(d, n_cls))
        self.net = nn.Sequential(*layers)
    def normalise(self, x): return (x - self.mu) / self.sd
    def forward(self, x, normalised=False):
        return self.net(x if normalised else self.normalise(x))

def ce_loss(logits, y, cw=None, ls=0.0):
    # label-smoothed CE, each sample weighted by the weight of its TRUE class.
    # (F.cross_entropy(weight=..., label_smoothing=...) also multiplies the smoothing term by every class weight,
    #  which blows up with very rare classes -> the model just predicts rare classes.)
    logp = logits.log_softmax(-1)
    l = (1 - ls) * -logp.gather(1, y[:, None]).squeeze(1) + ls * -logp.mean(-1)
    if cw is None: return l.mean()
    w = cw[y]; return (w * l).sum() / w.sum()

def set_seed(s):
    np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

@torch.no_grad()
def predict(model, X, bs=4096):
    model.eval(); X = torch.as_tensor(X, device=DEV)
    return torch.cat([model(X[i:i+bs]) for i in range(0, len(X), bs)]).softmax(-1).cpu().numpy()

def scores(y, p):
    return dict(acc=accuracy_score(y, p), macro_f1=f1_score(y, p, average='macro'),
                bal_acc=balanced_accuracy_score(y, p))

LOG_PRIOR = np.log(PRIOR_VAL + 1e-9)
def adjust(P, tau):
    # logit adjustment for prior shift: balanced train -> val/test prior
    return (np.log(P + 1e-9) + tau * LOG_PRIOR).argmax(1)

def tune_tau(P_val, y_val, taus=np.arange(0, 2.01, 0.1)):
    f = [f1_score(y_val, adjust(P_val, t), average='macro') for t in taus]
    return float(taus[int(np.argmax(f))]), max(f)

def train_mlp(X, y, idx, cfg, seed=0, verbose=True):
    set_seed(seed)
    Xtr, ytr = X[idx['train']], y[idx['train']]
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-6
    model = MLP(X.shape[1], C, cfg['hidden'], cfg['dropout'], mu, sd).to(DEV)
    Xt = model.normalise(torch.as_tensor(Xtr, device=DEV)); yt = torch.as_tensor(ytr, device=DEV)

    cw = None
    if cfg['class_weight']:
        cnt = np.bincount(ytr, minlength=C).astype(np.float32)
        cw = torch.as_tensor(cnt.sum() / (C * np.maximum(cnt, 1)), device=DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    steps = int(np.ceil(len(Xt) / cfg['batch_size']))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, cfg['lr'], epochs=cfg['epochs'], steps_per_epoch=steps)

    hist, best, best_state, wait = [], -1, None, 0
    for ep in range(cfg['epochs']):
        model.train(); perm = torch.randperm(len(Xt), device=DEV); tot = 0.0
        for b in range(steps):
            j = perm[b*cfg['batch_size']:(b+1)*cfg['batch_size']]
            if len(j) < 2: continue                                  # BatchNorm needs >1 sample
            xb, yb = Xt[j], yt[j]
            if cfg['input_noise']: xb = xb + cfg['input_noise'] * torch.randn_like(xb)
            if cfg['mixup_alpha']:
                lam = float(np.random.beta(cfg['mixup_alpha'], cfg['mixup_alpha'])); k = torch.randperm(len(xb), device=DEV)
                out = model(lam * xb + (1 - lam) * xb[k], normalised=True)
                loss = lam * ce_loss(out, yb, cw, cfg['label_smoothing']) + (1 - lam) * ce_loss(out, yb[k], cw, cfg['label_smoothing'])
            else:
                loss = ce_loss(model(xb, normalised=True), yb, cw, cfg['label_smoothing'])
            opt.zero_grad(); loss.backward(); opt.step(); sched.step(); tot += loss.item() * len(j)
        pv = predict(model, X[idx['val']])
        vl = F.nll_loss(torch.log(torch.as_tensor(pv) + 1e-9), torch.as_tensor(y[idx['val']])).item()
        sv = scores(y[idx['val']], pv.argmax(1))
        hist.append(dict(epoch=ep, train_loss=tot / len(Xt), val_loss=vl, **{f'val_{k}': v for k, v in sv.items()}))
        if sv['macro_f1'] > best:
            best, best_state, wait = sv['macro_f1'], copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= cfg['patience']: break
        if verbose and ep % 10 == 0:
            print(f"ep {ep:3d}  train {hist[-1]['train_loss']:.3f}  val_loss {vl:.3f}  val_f1 {sv['macro_f1']:.3f}  val_acc {sv['acc']:.3f}")
    model.load_state_dict(best_state)
    if verbose: print(f'best val macro-F1 {best:.4f} @ epoch {int(np.argmax([h["val_macro_f1"] for h in hist]))}')
    return model, pd.DataFrame(hist)

## 4. Hyper-parameter random search on **val**
`N_TRIALS` random configs per feature set (same configs for every feature), scored by val macro-F1 **after** logit adjustment (τ tuned on val). Test is never touched here. Set `RUN_SWEEP=False` to skip.

In [ ]:
SPACE = dict(hidden=[[512, 256], [1024, 512], [1024, 512, 256], [2048, 1024], [1024, 1024, 512]],
             dropout=[0.2, 0.3, 0.4, 0.5], lr=[1e-3, 5e-4, 3e-4], weight_decay=[1e-2, 5e-2, 1e-1],
             mixup_alpha=[0.0, 0.2, 0.4], input_noise=[0.0, 0.05, 0.1], label_smoothing=[0.0, 0.1],
             class_weight=[False, True])
rs = np.random.RandomState(123)
TRIALS = [{k: v[rs.randint(len(v))] for k, v in SPACE.items()} for _ in range(N_TRIALS)]

BEST_CFG = {name: dict(CFG) for name in FEATURES}
if RUN_SWEEP:
    sweep_rows = []
    for name, X in FEATURES.items():
        for t, trial in enumerate(TRIALS):
            cfg = {**CFG, **trial}; t0 = time.time()
            m, h = train_mlp(X, y_all, IDX, cfg, seed=0, verbose=False)
            Pv = predict(m, X[IDX['val']]); yv = y_all[IDX['val']]
            tau, f_adj = tune_tau(Pv, yv)
            sweep_rows.append(dict(feature=name, trial=t, val_f1_raw=f1_score(yv, Pv.argmax(1), average='macro'),
                                   val_f1_adj=f_adj, tau=tau, epochs=len(h), cfg=json.dumps(trial)))
            r = sweep_rows[-1]
            print(f"{name:12s} #{t:2d} raw {r['val_f1_raw']:.4f} adj {f_adj:.4f} (tau {tau:.1f}) {time.time()-t0:4.1f}s {trial}")
    sweep = pd.DataFrame(sweep_rows).sort_values(['feature', 'val_f1_adj'], ascending=[True, False])
    sweep.to_csv(OUT_DIR/'sweep.csv', index=False)
    for name in FEATURES:
        BEST_CFG[name].update(json.loads(sweep[sweep.feature == name].iloc[0].cfg))
    with pd.option_context('display.max_colwidth', 200):
        display(sweep.groupby('feature').head(3))
print(json.dumps(BEST_CFG, indent=1))

## 5. Baseline — Logistic Regression (same features)
If the MLP can't beat a linear model on the same input, the extra layers aren't buying anything.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

rows, PROBS = [], {}
for name, X in FEATURES.items():
    t0 = time.time()
    lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=0.1))
    lr.fit(X[IDX['train']], y_all[IDX['train']])
    Pv, Pt = lr.predict_proba(X[IDX['val']]), lr.predict_proba(X[IDX['test']])
    tau, _ = tune_tau(Pv, y_all[IDX['val']])
    for split, P in (('val', Pv), ('test', Pt)):
        rows.append(dict(feature=name, model='logreg', split=split, seed=0, **scores(y_all[IDX[split]], P.argmax(1))))
        rows.append(dict(feature=name, model='logreg+LA', split=split, seed=0, **scores(y_all[IDX[split]], adjust(P, tau))))
    print(f'{name:12s} logreg done ({time.time()-t0:.1f}s), tau={tau:.1f}')
pd.DataFrame(rows).pivot_table(index=['feature', 'model'], columns='split', values=['acc', 'macro_f1']).round(4)

## 6. Train MLP — both methods, several seeds

In [ ]:
HIST = {}
for name, X in FEATURES.items():
    for seed in SEEDS:
        print(f'\n=== {name} | seed {seed} ===')
        t0 = time.time()
        model, hist = train_mlp(X, y_all, IDX, BEST_CFG[name], seed=seed, verbose=False)
        for split in ('val', 'test'):
            pr = predict(model, X[IDX[split]])
            rows.append(dict(feature=name, model='mlp', split=split, seed=seed, **scores(y_all[IDX[split]], pr.argmax(1))))
            PROBS[(name, seed, split)] = pr
        print(f"best val macro-F1 {hist.val_macro_f1.max():.4f} @ epoch {int(hist.val_macro_f1.idxmax())}")
        HIST[(name, seed)] = hist
        torch.save({'state_dict': model.state_dict(), 'cfg': BEST_CFG[name], 'd_in': X.shape[1], 'classes': CLASSES,
                    'log_prior_val': LOG_PRIOR},
                   OUT_DIR/f'mlp_{name}_seed{seed}.pt')
        print(f'{time.time()-t0:.1f}s')

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
for (name, seed), h in HIST.items():
    if seed != SEEDS[0]: continue
    ls = {'whisper': '-', 'mel': '--'}.get(name, ':'); lab = name
    ax[0].plot(h.epoch, h.train_loss, ls, label=lab); ax[1].plot(h.epoch, h.val_loss, ls, label=lab)
    ax[2].plot(h.epoch, h.val_macro_f1, ls, label=lab)
for a, t in zip(ax, ['train loss', 'val loss', 'val macro-F1']): a.set_title(t); a.set_xlabel('epoch')
ax[2].legend(fontsize=7); plt.tight_layout(); plt.savefig(OUT_DIR/'curves.png', dpi=120)

## 7. Results — Whisper vs Mel

Rows: `mlp` = single models (mean ± std over seeds) · `mlp-ens` = average of the seed models' probabilities · `+LA` = logit adjustment with τ tuned on val.

In [ ]:
yv, yt = y_all[IDX['val']], y_all[IDX['test']]
FINAL_PRED, TAU = {}, {}
for name in FEATURES:
    Pv = np.mean([PROBS[(name, s, 'val')] for s in SEEDS], 0); Pt = np.mean([PROBS[(name, s, 'test')] for s in SEEDS], 0)
    TAU[name], _ = tune_tau(Pv, yv)
    for split, y, P in (('val', yv, Pv), ('test', yt, Pt)):
        rows.append(dict(feature=name, model='mlp-ens', split=split, seed=-1, **scores(y, P.argmax(1))))
        rows.append(dict(feature=name, model='mlp-ens+LA', split=split, seed=-1, **scores(y, adjust(P, TAU[name]))))
    FINAL_PRED[name] = adjust(Pt, TAU[name])
print('tau per feature:', TAU)

res = pd.DataFrame(rows); res.to_csv(OUT_DIR/'results_all.csv', index=False)
summary = (res.groupby(['feature', 'model', 'split'])[['acc', 'macro_f1', 'bal_acc']]
              .agg(['mean', 'std']).round(4))
summary.to_csv(OUT_DIR/'results_summary.csv')
test_tbl = res[res.split == 'test'].groupby(['feature', 'model'])[['acc', 'macro_f1', 'bal_acc']].mean().round(4)
print(test_tbl.to_string())     # also printed so it shows in the run log
test_tbl

In [ ]:
# per-class report + confusion matrix (test, final = seed-ensemble + logit adjustment)
fig, ax = plt.subplots(1, len(FEATURES), figsize=(6 * len(FEATURES), 5.5))
for a, name in zip(np.atleast_1d(ax), FEATURES):
    p = FINAL_PRED[name]
    print(f'\n##### {name} (ensemble + LA, tau={TAU[name]:.1f})  macro-F1 = {f1_score(yt, p, average="macro"):.4f}')
    print(classification_report(yt, p, labels=range(C), target_names=CLASSES, digits=3, zero_division=0))
    cm = confusion_matrix(yt, p, labels=range(C), normalize='true')
    a.imshow(cm, cmap='Blues', vmin=0, vmax=1); a.set_title(f'{name} — test (row-normalised)')
    a.set_xticks(range(C)); a.set_xticklabels(CLASSES, rotation=90, fontsize=7)
    a.set_yticks(range(C)); a.set_yticklabels(CLASSES, fontsize=7)
    a.set_xlabel('predicted'); a.set_ylabel('true')
plt.tight_layout(); plt.savefig(OUT_DIR/'confusion.png', dpi=120)

In [ ]:
# load a saved model later:
# ck = torch.load(OUT_DIR/'mlp_whisper_seed0.pt', map_location=DEV)
# m = MLP(ck['d_in'], len(ck['classes']), ck['cfg']['hidden'], ck['cfg']['dropout']).to(DEV)
# m.load_state_dict(ck['state_dict']); probs = predict(m, X_whisper[IDX['test']])